# 03 – Bereinigung & SQLite-Datenbank
**Verantwortlich:** Martin

**Ziel:**
1. Die Rohdaten aus der API (`data/spiele_roh.csv`) prüfen und bereinigen
2. Zusätzliche Spalten berechnen (Datum, Uhrzeit, Wochentag, Tore, Heimsieg …)
3. Alles in einer **SQLite-Datenbank** speichern (`data/football.db`)
4. **SQL-Abfragen aus Python** ausführen (Mindestanforderung 3)

Ausführen: oben auf **„Run All“** klicken.

In [1]:
import sqlite3
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

df_roh = pd.read_csv("data/spiele_roh.csv")
print(f"Rohdaten: {len(df_roh)} Spiele, {df_roh.shape[1]} Spalten")
df_roh.head()

Rohdaten: 3198 Spiele, 13 Spalten


,match_id,liga,saison,spieltag,anstoss_utc,status,heim_id,heim_team,gast_id,gast_team,tore_heim,tore_gast,resultat
0,435943,Premier League,2023/24,1,2023-08-11T19:00:00Z,FINISHED,328,Burnley FC,65,Manchester City FC,0,3,A
1,435944,Premier League,2023/24,1,2023-08-12T12:00:00Z,FINISHED,57,Arsenal FC,351,Nottingham Forest FC,2,1,H
2,435945,Premier League,2023/24,1,2023-08-12T14:00:00Z,FINISHED,1044,AFC Bournemouth,563,West Ham United FC,1,1,D
3,435946,Premier League,2023/24,1,2023-08-12T14:00:00Z,FINISHED,397,Brighton & Hove Albion FC,389,Luton Town FC,4,1,H
4,435947,Premier League,2023/24,1,2023-08-12T14:00:00Z,FINISHED,62,Everton FC,63,Fulham FC,0,1,A


## 1. Datenprüfung
Fehlende Werte, doppelte Spiele und Status der Spiele.

In [2]:
print("Fehlende Werte pro Spalte:")
print(df_roh.isna().sum())

print("\nDoppelte match_id:", df_roh["match_id"].duplicated().sum())

print("\nStatus der Spiele:")
print(df_roh["status"].value_counts())

print("\nSpiele, die nicht normal beendet wurden:")
df_roh[df_roh["status"] != "FINISHED"][["liga", "saison", "anstoss_utc", "heim_team", "gast_team",
                                        "tore_heim", "tore_gast", "status"]]

Fehlende Werte pro Spalte:
match_id       0
liga           0
saison         0
spieltag       0
anstoss_utc    0
status         0
heim_id        0
heim_team      0
gast_id        0
gast_team      0
tore_heim      0
tore_gast      0
resultat       0
dtype: int64

Doppelte match_id: 0

Status der Spiele:
status
FINISHED    3197
AWARDED        1
Name: count, dtype: int64

Spiele, die nicht normal beendet wurden:


,liga,saison,anstoss_utc,heim_team,gast_team,tore_heim,tore_gast,status
1566,Bundesliga,2024/25,2024-12-14T14:30:00Z,1. FC Union Berlin,VfL Bochum 1848,0,2,AWARDED


## 2. Bereinigung
- Nur Spiele mit Status **FINISHED** behalten (Spiele mit Status *AWARDED* wurden am grünen Tisch entschieden → nicht normal gespielt)
- Doppelte Spiele entfernen (falls vorhanden)
- Anstosszeit in ein Datumsformat umwandeln (Zeitzone **UTC**, wie später beim Wetter)

In [3]:
protokoll = []   # Protokoll für die Dokumentation der Bereinigung
protokoll.append(("Rohdaten aus API", len(df_roh)))

df = df_roh[df_roh["status"] == "FINISHED"].copy()
protokoll.append(("nach Entfernen nicht beendeter Spiele (z.B. AWARDED)", len(df)))

df = df.drop_duplicates(subset="match_id")
protokoll.append(("nach Entfernen doppelter Spiele", len(df)))

df["anstoss_utc"] = pd.to_datetime(df["anstoss_utc"], utc=True)
df["tore_heim"] = df["tore_heim"].astype(int)
df["tore_gast"] = df["tore_gast"].astype(int)

for schritt, anzahl in protokoll:
    print(f"{anzahl:>6}  {schritt}")
print(f"\nEntfernt total: {protokoll[0][1] - protokoll[-1][1]} Spiele")

  3198  Rohdaten aus API
  3197  nach Entfernen nicht beendeter Spiele (z.B. AWARDED)
  3197  nach Entfernen doppelter Spiele

Entfernt total: 1 Spiele


## 3. Zusätzliche Spalten
| Spalte | Bedeutung |
|---|---|
| `datum` | Spieldatum (UTC) |
| `stunde_utc` | Anstoss-Stunde in UTC (für die Wetterabfrage) |
| `wochentag`, `wochenende` | Wochentag, Wochenende ja/nein |
| `monat` | Monat (1–12) |
| `tore_total` | Tore im Spiel |
| `tordifferenz` | Tore Heim − Tore Gast |
| `heimsieg` | 1 = Heimsieg, 0 = sonst (Zielvariable für das Modell) |

In [4]:
df["datum"]        = df["anstoss_utc"].dt.strftime("%Y-%m-%d")
df["stunde_utc"]   = df["anstoss_utc"].dt.hour
df["wochentag"]    = df["anstoss_utc"].dt.day_name()
df["wochenende"]   = df["anstoss_utc"].dt.dayofweek.isin([5, 6]).astype(int)
df["monat"]        = df["anstoss_utc"].dt.month
df["tore_total"]   = df["tore_heim"] + df["tore_gast"]
df["tordifferenz"] = df["tore_heim"] - df["tore_gast"]
df["heimsieg"]     = (df["resultat"] == "H").astype(int)

# Anstosszeit als Text speichern (SQLite kennt keinen eigenen Datums-Typ)
df["anstoss_utc"] = df["anstoss_utc"].dt.strftime("%Y-%m-%d %H:%M:%S")

df.head()

C:\Users\filip\AppData\Local\Temp\ipykernel_25696\2427819098.py:4: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["wochenende"]   = df["anstoss_utc"].dt.dayofweek.isin([5, 6]).astype(int)


,match_id,liga,saison,spieltag,anstoss_utc,status,heim_id,heim_team,gast_id,gast_team,tore_heim,tore_gast,resultat,datum,stunde_utc,wochentag,wochenende,monat,tore_total,tordifferenz,heimsieg
0,435943,Premier League,2023/24,1,2023-08-11 19:00:00,FINISHED,328,Burnley FC,65,Manchester City FC,0,3,A,2023-08-11,19,Friday,0,8,3,-3,0
1,435944,Premier League,2023/24,1,2023-08-12 12:00:00,FINISHED,57,Arsenal FC,351,Nottingham Forest FC,2,1,H,2023-08-12,12,Saturday,1,8,3,1,1
2,435945,Premier League,2023/24,1,2023-08-12 14:00:00,FINISHED,1044,AFC Bournemouth,563,West Ham United FC,1,1,D,2023-08-12,14,Saturday,1,8,2,0,0
3,435946,Premier League,2023/24,1,2023-08-12 14:00:00,FINISHED,397,Brighton & Hove Albion FC,389,Luton Town FC,4,1,H,2023-08-12,14,Saturday,1,8,5,3,1
4,435947,Premier League,2023/24,1,2023-08-12 14:00:00,FINISHED,62,Everton FC,63,Fulham FC,0,1,A,2023-08-12,14,Saturday,1,8,1,-1,0


## 4. SQLite-Datenbank erstellen
Zwei Tabellen:
- **teams** – jedes Team einmal (ID, Name, Liga)
- **matches** – jedes Spiel einmal, Teams nur über ihre ID (damit wir später mit `JOIN` arbeiten)

In [5]:
teams = pd.concat([
    df[["heim_id", "heim_team", "liga"]].rename(columns={"heim_id": "team_id", "heim_team": "team_name"}),
    df[["gast_id", "gast_team", "liga"]].rename(columns={"gast_id": "team_id", "gast_team": "team_name"}),
]).drop_duplicates(subset="team_id").sort_values(["liga", "team_name"])

matches = df[["match_id", "liga", "saison", "spieltag", "anstoss_utc", "datum", "stunde_utc",
              "wochentag", "wochenende", "monat", "heim_id", "gast_id",
              "tore_heim", "tore_gast", "tore_total", "tordifferenz", "resultat", "heimsieg"]]

con = sqlite3.connect("data/football.db")
teams.to_sql("teams", con, if_exists="replace", index=False)
matches.to_sql("matches", con, if_exists="replace", index=False)
con.execute("CREATE INDEX IF NOT EXISTS idx_matches_liga ON matches(liga, saison)")
con.commit()

print(f"Tabelle teams:   {len(teams)} Teams")
print(f"Tabelle matches: {len(matches)} Spiele")
print("Gespeichert in: data/football.db")

Tabelle teams:   72 Teams
Tabelle matches: 3197 Spiele
Gespeichert in: data/football.db


## 5. SQL-Abfragen aus Python
### 5.1 Anzahl Spiele pro Liga und Saison

In [6]:
pd.read_sql_query("""
    SELECT liga, saison, COUNT(*) AS spiele
    FROM matches
    GROUP BY liga, saison
    ORDER BY liga, saison
""", con)

,liga,saison,spiele
0,Bundesliga,2023/24,306
1,Bundesliga,2024/25,305
2,Bundesliga,2025/26,306
3,La Liga,2023/24,380
4,La Liga,2024/25,380
5,La Liga,2025/26,380
6,Premier League,2023/24,380
7,Premier League,2024/25,380
8,Premier League,2025/26,380


### 5.2 Heimsieg-, Unentschieden- und Auswärtssieg-Quote pro Liga
Ein erster Blick auf den **Heimvorteil** in den drei Ligen.

In [7]:
pd.read_sql_query("""
    SELECT liga,
           COUNT(*)                                                       AS spiele,
           ROUND(100.0 * AVG(CASE WHEN resultat = 'H' THEN 1 ELSE 0 END), 1) AS heimsieg_pct,
           ROUND(100.0 * AVG(CASE WHEN resultat = 'D' THEN 1 ELSE 0 END), 1) AS unentschieden_pct,
           ROUND(100.0 * AVG(CASE WHEN resultat = 'A' THEN 1 ELSE 0 END), 1) AS auswaertssieg_pct,
           ROUND(AVG(tore_total), 2)                                      AS tore_pro_spiel
    FROM matches
    GROUP BY liga
    ORDER BY heimsieg_pct DESC
""", con)

,liga,spiele,heimsieg_pct,unentschieden_pct,auswaertssieg_pct,tore_pro_spiel
0,La Liga,1140,45.8,26.1,28.2,2.65
1,Premier League,1140,43.2,24.5,32.4,2.99
2,Bundesliga,917,42.1,25.4,32.5,3.20


### 5.3 Heimsieg-Quote nach Monat
Erster Hinweis auf das Wetter: Ist der Heimvorteil im **Winter** anders als im Sommer/Herbst?

In [8]:
pd.read_sql_query("""
    SELECT liga, monat,
           COUNT(*)                       AS spiele,
           ROUND(100.0 * AVG(heimsieg), 1) AS heimsieg_pct
    FROM matches
    GROUP BY liga, monat
    ORDER BY liga, monat
""", con)

,liga,monat,spiele,heimsieg_pct
0,Bundesliga,1,106,46.2
1,Bundesliga,2,108,41.7
2,Bundesliga,3,102,31.4
3,Bundesliga,4,108,38.0
4,Bundesliga,5,81,35.8
5,Bundesliga,8,52,42.3
6,Bundesliga,9,90,41.1
7,Bundesliga,10,84,48.8
8,Bundesliga,11,96,45.8
9,Bundesliga,12,90,51.1


### 5.4 Stärkste Heimteams (mit JOIN)
Teams mit der höchsten Heimsieg-Quote (mind. 30 Heimspiele).

In [9]:
pd.read_sql_query("""
    SELECT t.team_name, m.liga,
           COUNT(*)                         AS heimspiele,
           ROUND(100.0 * AVG(m.heimsieg), 1) AS heimsieg_pct
    FROM matches m
    JOIN teams t ON m.heim_id = t.team_id
    GROUP BY t.team_name, m.liga
    HAVING COUNT(*) >= 30
    ORDER BY heimsieg_pct DESC
    LIMIT 10
""", con)

,team_name,liga,heimspiele,heimsieg_pct
0,FC Barcelona,La Liga,57,84.2
1,Real Madrid CF,La Liga,57,84.2
2,FC Bayern München,Bundesliga,51,82.4
3,Club Atlético de Madrid,La Liga,57,78.9
4,Arsenal FC,Premier League,57,71.9
5,Manchester City FC,Premier League,57,71.9
6,Liverpool FC,Premier League,57,68.4
7,Borussia Dortmund,Bundesliga,51,66.7
8,Bayer 04 Leverkusen,Bundesliga,51,64.7
9,VfB Stuttgart,Bundesliga,51,62.7


In [10]:
con.close()

## Für die Dokumentation (Anforderung 2)
- Rohdaten: Anzahl Spiele aus der API (siehe Protokoll oben)
- Entfernt: Spiele mit Status ≠ FINISHED (z.B. 1 Spiel *AWARDED*, am grünen Tisch entschieden)
- Anstosszeiten einheitlich in **UTC** → passt direkt zur Wetter-API (Open-Meteo, `timezone=UTC`)

**Nächste Schritte:** Hebron ordnet jedem Heimteam (`teams`-Tabelle) ein Stadion zu und holt das Wetter → neue Tabellen `stadiums` und `weather` in derselben Datenbank.